# 4.5 GBDT+LR：树叶特征与概率校准

> 阅读版与 Web 应用内容一致；实验数值来自本 Notebook 的已执行输出。

## Goal

单独掌握经典两阶段 CTR 模型：用 XGBoost 学条件规则，将叶节点 one-hot 后训练 LR，并复现完整在线推理链。

## Setup

本 Notebook 的默认真实数据是 **MovieLens 评分派生标签（bounded teaching view）；full 档使用 Criteo_x1 官方 7:2:1 切分**。`smoke` 档读取仓库内可审计的确定性切片，`full` 档扩大到官方完整文件；两档都不制造交互、曝光、标签或行为序列。切片规则、源地址、哈希与许可记录在 `data/README.md` 及对应数据目录。

**主要资料：** [Practical Lessons from Predicting Clicks on Ads at Facebook](https://research.facebook.com/publications/practical-lessons-from-predicting-clicks-on-ads-at-facebook/)

In [1]:
from pathlib import Path
import os, sys, json
import torch
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACT_ROOT = Path(os.environ.get("RECSYS_ARTIFACT_ROOT", PROJECT_ROOT)).expanduser().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("RECSYS_PROFILE", "full")
PROFILE = os.environ["RECSYS_PROFILE"]
CUDA_AVAILABLE = torch.cuda.is_available()
DATASET_KEY = "criteo-x1"
# Setup 只声明执行边界。完整数据由章节 runner 在 Train & Inference 单元按需读取，
# 避免仅打开 Notebook 就解析数千万行文件。
REAL_DATASET = {
    "dataset": DATASET_KEY,
    "profile": PROFILE,
    "loading": "lazy: chapter runner owns loading and returns executed provenance",
    "randomly_fabricated_rows": 0,
}
print({"profile": PROFILE, "project_root": str(PROJECT_ROOT), "artifact_root": str(ARTIFACT_ROOT), "dataset_boundary": REAL_DATASET,
       "cuda_available": CUDA_AVAILABLE,
       "cuda_device": torch.cuda.get_device_name(0) if CUDA_AVAILABLE else None})
assert REAL_DATASET["randomly_fabricated_rows"] == 0

{'profile': 'smoke', 'project_root': '<ARTIFACT_ROOT>', 'artifact_root': '<ARTIFACT_ROOT>', 'dataset_boundary': {'dataset': 'criteo-x1', 'profile': 'smoke', 'loading': 'lazy: chapter runner owns loading and returns executed provenance', 'randomly_fabricated_rows': 0}, 'cuda_available': False, 'cuda_device': None}


## 来源论文与阅读提示

**He et al. (2014), Practical Lessons from Predicting Clicks on Ads at Facebook** 的关键工程判断是把 GBDT 当作监督式特征变换：每棵树的叶节点代表一组条件规则，叶 one-hot 再进入 LR。它不是端到端模型，因此训练/服务必须同时版本化树、叶编码器和 LR。

### 公式怎么读（直觉版）

一棵决策树像连续做选择题：“是否晚间？”“是否移动设备？”最终落到一个叶子。把每棵树落到的叶子变成 0/1 开关，再由 LR 做加权求和。Sigmoid 最后把任意实数分数压到 0～1，变成点击概率；为什么要用 LogLoss 训练，可先看 **3.0 推荐算法数学基础** 的概率图。

In [2]:
# Bounded teaching view：无论正式 PROFILE 是 smoke 还是 full，这一段公式演示都只读取
# 仓库内的 MovieLens latest-small 确定性小视图。正式实验由后面的 chapter runner 单独加载。
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, roc_auc_score, log_loss
from recsys_lab import data as data_tools
from recsys_lab.data import leave_last_out, movielens_provenance

SEED = 2026
torch.manual_seed(SEED)
_formal_profile = os.environ["RECSYS_PROFILE"]
try:
    os.environ["RECSYS_PROFILE"] = "smoke"
    data_tools._load_cached.cache_clear()
    ratings, movies = data_tools.load_movielens(max_users=48, max_items=360, min_user_events=12)
finally:
    os.environ["RECSYS_PROFILE"] = _formal_profile
    data_tools._load_cached.cache_clear()

train_ratings, test_ratings = leave_last_out(ratings)
n_users, n_items = ratings.user_id.nunique(), ratings.item_id.nunique()
TEACHING_DATASET = movielens_provenance(ratings)
assert TEACHING_DATASET["profile"] == "smoke"
assert TEACHING_DATASET["randomly_fabricated_rows"] == 0
print({
    "mode": "bounded teaching view (not the formal experiment)",
    "rows": len(ratings), "users": n_users, "items": n_items,
    "train_rows": len(train_ratings), "test_rows": len(test_ratings),
    "source": TEACHING_DATASET["source"], "fabricated_rows": 0,
})
display(ratings[["userId", "movieId", "rating", "timestamp", "title", "genres"]].head(8))

{'mode': 'bounded teaching view (not the formal experiment)', 'rows': 10487, 'users': 48, 'items': 360, 'train_rows': 10439, 'test_rows': 48, 'source': 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip', 'fabricated_rows': 0}


,userId,movieId,rating,timestamp,title,genres
0,140,318,4.0,942840891,"Shawshank Redemption, The (1994)",Crime|Drama
1,140,527,5.0,942840891,Schindler's List (1993),Drama|War
2,140,1221,3.0,942840891,"Godfather: Part II, The (1974)",Crime|Drama
3,140,50,3.0,942840991,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
4,140,595,5.0,942840991,Beauty and the Beast (1991),Animation|Children|Fantasy|Musical|Romance|IMAX
5,140,1198,4.0,942841170,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure
6,140,2028,5.0,942841170,Saving Private Ryan (1998),Action|Drama|War
7,140,1721,4.0,942841219,Titanic (1997),Drama|Romance


---

## 4. 因子分解机（FM）：为稀疏特征学习二阶交互

### 4.1 为什么 MF 不够？

**通用先修：** [3.2 样本、特征与标签](/notebooks/3_2_data_ml_basics#observation-label) · [3.3 逐元素与点积](/notebooks/3_3_linear_algebra#elementwise-dot) · [3.6 二元交叉熵](/notebooks/3_6_information_theory#cross-entropy-kl)<br>
**本论文新增数学：** 用共享隐向量参数化任意稀疏特征的二阶交互，并把 $O(n^2k)$ 的逐对计算化成 $O(nk)$ 恒等式。

CTR 排序不只有 `user_id` 和 `item_id`，还会有设备、小时、地域、品类、价格等上下文。直接为每一对特征做 one-hot 交叉会导致参数爆炸；普通 LR 又只能做加法，无法表达“某用户群在晚间更喜欢某品类”。

FM 的输入是一个超长稀疏向量 $x\in\mathbb R^n$：每个特征占一格，one-hot 类别特征取 0/1，连续特征取实数值。模型为每个特征 $i$ 学一个 $k$ 维隐向量 $v_i\in\mathbb R^k$，用内积 $\langle v_i,v_j\rangle=\sum_{f=1}^{k}v_{if}v_{jf}$ 共享所有二阶交叉统计：

$$
\hat y(x)=w_0+\sum_i w_ix_i+\sum_{i<j}\langle v_i,v_j\rangle x_ix_j
$$

三项分别是：$w_0$ 全局偏置；$\sum_i w_ix_i$ 一阶线性项（与 LR 相同）；$\sum_{i<j}\langle v_i,v_j\rangle x_ix_j$ 二阶交叉项——只有同时“打开”（$x_i,x_j$ 都非零）的特征对才参与。关键在于稀疏性：即使某对特征从未在训练集共同出现，它们各自与其他特征的共现也已把 $v_i,v_j$ 学好，交叉强度仍能由内积给出。

朴素计算二阶项要枚举全部 $\binom{n}{2}$ 对特征、每对做一次 $k$ 维内积，复杂度 $O(n^2k)$。利用“和的平方减去平方的和”恒等式可化为 $O(nk)$：

$$
\sum_{i<j}\langle v_i,v_j\rangle x_ix_j
=\frac12\sum_f\left[\left(\sum_i v_{if}x_i\right)^2-\sum_i v_{if}^2x_i^2\right]
$$

推导只需一步代数：把 $(\sum_i v_{if}x_i)^2$ 展开会得到所有有序对 $(i,j)$ 的乘积——包含 $i=j$ 的对角项，且每对 $i<j$ 出现两次；减去对角项 $\sum_i v_{if}^2x_i^2$ 再乘 $\tfrac12$ 去重，正好回到左边。右侧只需对每个因子维度 $f$ 各扫一遍特征。

**手算一遍。** 取 $k=2$、两个特征 $v_1=[1,2]$、$v_2=[0.5,-1]$，且 $x_1=x_2=1$。左边：$\langle v_1,v_2\rangle=1\times0.5+2\times(-1)=-1.5$。右边逐维算：$f=1$ 时 $(1+0.5)^2-(1^2+0.5^2)=2.25-1.25=1$；$f=2$ 时 $(2+(-1))^2-(2^2+(-1)^2)=1-5=-4$；合起来 $\tfrac12[1+(-4)]=-1.5$，与左边一致。

当特征只有 user one-hot 和 item one-hot 时，FM 的交叉项就退化为 MF；因此 FM 可以理解为 MF 对通用稀疏特征的扩展。

### 4.2 数据：评分阈值派生“高偏好”标签（不是曝光—点击日志）

MovieLens 记录的是用户主动提交的星级，**没有完整曝光分母**。这里把真实评分 `rating >= 4.0` 派生为高偏好标签、较低评分派生为 0，只用于演示 FM 的二分类、AUC 与 LogLoss 计算；变量名 `click` 是代码兼容别名，不代表真实广告点击。没有评分的 user–item 仍是未知，不能补成曝光未点击。真实 CTR 结论必须改用包含曝光与点击的日志。

In [3]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ctr = ratings.sort_values("timestamp").tail(5000).copy().reset_index(drop=True)
# `click` 是任务别名；标签由真实评分 rating >= 4.0 确定，不使用随机采样。
ctr["click"] = ctr["like"].astype(int)
split = int(len(ctr) * .8)  # 严格按真实 timestamp 切分
categorical = ["user_id", "item_id", "genre_id", "hour", "decade_id"]
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
train_sparse = encoder.fit_transform(ctr.loc[:split-1, categorical])
test_sparse = encoder.transform(ctr.loc[split:, categorical])
scaler = StandardScaler()
train_numeric = scaler.fit_transform(ctr.loc[:split-1, ["item_popularity", "user_activity"]])
test_numeric = scaler.transform(ctr.loc[split:, ["item_popularity", "user_activity"]])
fm_train_x = np.c_[train_sparse, train_numeric].astype("float32")
fm_test_x = np.c_[test_sparse, test_numeric].astype("float32")
ctr_train_y = ctr.loc[:split-1, "click"].to_numpy()
ctr_test_y = ctr.loc[split:, "click"].to_numpy()
print({"train": fm_train_x.shape, "test": fm_test_x.shape, "train_positive_rate": round(ctr_train_y.mean(), 3)})

{'train': (4000, 437), 'test': (1000, 437), 'train_positive_rate': np.float64(0.474)}


---

## 5. GBDT+LR：树负责发现规则，LR 负责组合与校准

### 5.1 核心思想

**通用先修：** [3.2 标签、曝光与校准边界](/notebooks/3_2_data_ml_basics#implicit-feedback) · [3.5 odds、logit 与校准](/notebooks/3_5_probability_statistics#likelihood-calibration) · [3.6 交叉熵与 Normalized Entropy](/notebooks/3_6_information_theory#softmax-temperature)<br>
**本论文新增数学：** GBDT 逐轮拟合损失的负梯度、用切分形成条件规则，再把叶节点当稀疏特征交给 LR；并按 Facebook 论文口径解释 NE。

Facebook 经典 CTR 工作把 GBDT 的每棵树视为一个自动特征生成器。样本落到某棵树的哪个叶子，代表它满足了一组条件，例如：

> `hour > 18 AND device = mobile AND price < 20`

将每棵树的叶节点编号 one-hot 后输入 LR：

$$
P(y=1\mid z)=\sigma(w_0+w^\top z)
$$

符号逐个看：$z$ 是所有树叶节点的稀疏指示向量——若有 $T$ 棵树、第 $t$ 棵有 $L_t$ 个叶子，$z$ 的长度就是 $\sum_t L_t$，且每棵树恰好贡献一个 1；$w$ 是 LR 为每个叶子学到的权重；$\sigma(a)=1/(1+e^{-a})$ 把任意实数压到 0～1，变成点击概率。

**手算一遍。** 设 2 棵树，第 1 棵 3 个叶子、第 2 棵 2 个叶子，$z$ 长度为 5。某样本落入第 1 棵的 1 号叶、第 2 棵的 2 号叶，则 $z=[1,0,0,0,1]$。若 $w=[0.8,-0.2,0.1,-0.4,0.9]$、$w_0=-0.5$，线性得分 $=-0.5+0.8+0.9=1.2$，$\sigma(1.2)\approx0.77$——模型预测该样本约有 77% 的点击概率。

GBDT 的“Boosting”是逐轮纠错。第 $m$ 轮已有总分 $F_{m-1}(x)$，新树拟合每个样本让损失下降最快的方向（负梯度，也叫伪残差）：

$$
g_i^{(m)}=-\left.\frac{\partial\ell(y_i,F(x_i))}{\partial F(x_i)}\right|_{F=F_{m-1}},
\qquad F_m(x)=F_{m-1}(x)+\eta h_m(x).
$$

平方误差时 $g_i=y_i-F_{m-1}(x_i)$，就是熟悉的“真实值减当前预测”；二分类 LogLoss 时仍是残差直觉，但准确说法是负梯度。建树时尝试“特征 $j$ 是否小于阈值 $t$”等问题，把样本分到左右两堆；选择能让两边负梯度更一致、目标下降最多的切分。连续多次切分后，一个叶子便代表一串 AND 条件，LR 再学习这些规则的稳定权重。

原论文的 **Normalized Entropy** 用常数基线熵归一化平均 LogLoss。令样本正率 $\bar y=\frac1N\sum_i y_i$：

$$
\mathrm{NE}=
\frac{-\frac1N\sum_i[y_i\log p_i+(1-y_i)\log(1-p_i)]}
{-\bar y\log\bar y-(1-\bar y)\log(1-\bar y)}.
$$

分母是所有样本都预测总体正率 $\bar y$ 时的 LogLoss，因此 NE=1 约等于只会报基准率，越低越好。论文报告树+LR 组合相对单模型的 NE 改进超过 3%，但这是 Facebook 内部广告数据口径下的曝光日志；本教程用 MovieLens **评分阈值派生标签**验证链路，不是曝光点击，该收益不可平移。两阶段训练还意味着树、叶编码器和 LR 不是端到端联合优化。

### 5.2 训练阶段一：XGBoost 学习树规则

In [4]:
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

tree_features = ["user_id", "item_id", "genre_id", "hour", "decade_id", "item_popularity", "user_activity"]
# GBDT 使用与 FM 相同的 one-hot 类别语义，避免把类别 ID 错当连续数值。
gbdt_train_x = fm_train_x
gbdt_test_x = fm_test_x

gbdt = XGBClassifier(
    n_estimators=80, max_depth=4, learning_rate=.05, subsample=.9,
    colsample_bytree=.9, eval_metric="logloss", random_state=SEED, n_jobs=1
)
gbdt.fit(gbdt_train_x, ctr_train_y)
train_leaf = gbdt.apply(gbdt_train_x)
test_leaf = gbdt.apply(gbdt_test_x)
print({"trees": train_leaf.shape[1], "train_leaf_matrix": train_leaf.shape})

{'trees': 80, 'train_leaf_matrix': (4000, 80)}


### 5.3 训练阶段二：叶节点 one-hot 后训练 LR

In [5]:
leaf_encoder = OneHotEncoder(handle_unknown="ignore")
train_leaf_onehot = leaf_encoder.fit_transform(train_leaf)
test_leaf_onehot = leaf_encoder.transform(test_leaf)

leaf_lr = LogisticRegression(max_iter=500, C=.7)
leaf_lr.fit(train_leaf_onehot, ctr_train_y)
gbdt_lr_probability = leaf_lr.predict_proba(test_leaf_onehot)[:, 1]

gbdt_lr_auc = float(roc_auc_score(ctr_test_y, gbdt_lr_probability))
gbdt_lr_logloss = float(log_loss(ctr_test_y, gbdt_lr_probability))
print({"leaf_onehot_dim": train_leaf_onehot.shape[1], "GBDT+LR AUC": round(gbdt_lr_auc, 4), "GBDT+LR LogLoss": round(gbdt_lr_logloss, 4)})

{'leaf_onehot_dim': 1042, 'GBDT+LR AUC': 0.6589, 'GBDT+LR LogLoss': 0.7057}


<ARTIFACT_ROOT>/.venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: divide by zero encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
<ARTIFACT_ROOT>/.venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: overflow encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
<ARTIFACT_ROOT>/.venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: invalid value encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)


### 5.4 推理：同一条样本必须依次经过两阶段

In [6]:
def predict_gbdt_lr(frame):
    sparse_part = encoder.transform(frame[categorical])
    numeric_part = scaler.transform(frame[["item_popularity", "user_activity"]])
    encoded_features = np.c_[sparse_part, numeric_part].astype("float32")
    leaf = gbdt.apply(encoded_features)
    leaf_onehot = leaf_encoder.transform(leaf)
    return leaf_lr.predict_proba(leaf_onehot)[:, 1]

online_batch = ctr.loc[split:split+4].copy()
online_batch["p_click"] = predict_gbdt_lr(online_batch)
display(online_batch[tree_features + ["click", "p_click"]])

,user_id,item_id,genre_id,hour,decade_id,item_popularity,user_activity,click,p_click
4000,40,210,8,1,6,30.0,218.0,0,0.250832
4001,40,261,5,1,7,26.0,218.0,1,0.739649
4002,40,358,0,1,7,27.0,218.0,0,0.327755
4003,40,111,4,1,4,27.0,218.0,0,0.224486
4004,40,303,7,1,7,28.0,218.0,0,0.607288


### 5.5 结果讨论与边界

**优点**：连续变量无需手工分桶；树能发现条件组合；LR 服务成熟、输出易校准；对中等规模表格特征很强。  
**缺点**：两阶段可能失配；叶节点空间随树数膨胀；高基数 ID 容易过拟合；树规则会随分布漂移而陈旧；难以表达长行为序列。  
**工业注意**：训练与服务必须共享完全相同的树版本、叶编码器和缺失值处理；监控 unseen leaf、特征漂移与概率校准。

与 FM 的关键区别：FM 假设交互可由低秩内积共享；GBDT+LR 假设有效模式可由一组树规则离散化。两者没有绝对胜负，取决于稀疏度、特征类型、数据量和延迟预算。

## Train & Inference（正式实验）

下方唯一的正式指标入口调用 `chapter_code/<slug>/train.py`。runner 会按 `PROFILE`
惰性加载对应完整/CI 数据、执行内存受控训练与评测，并持续打印阶段进度。上面的
bounded teaching view 只用于手算和理解代码，不会写入章节结果。

In [7]:
from importlib import import_module
from recsys_lab.runtime import print_progress

chapter_train = import_module("chapter_code.4_5_gbdt_lr.train")
result = chapter_train.train_and_evaluate(
    epochs=8 if PROFILE == "smoke" else 8,
    progress=print_progress,
)
REAL_DATASET = result["dataset"]
_executed_fabricated_rows = (
    REAL_DATASET.get("randomly_fabricated_rows", result.get("randomly_fabricated_rows"))
    if isinstance(REAL_DATASET, dict)
    else result.get("randomly_fabricated_rows")
)
assert _executed_fabricated_rows == 0
print({"profile": PROFILE, "executed_dataset": REAL_DATASET})
print({key: value for key, value in result.items() if key not in {"dataset", "loss_curve"}})

[data_prepare] 0/1 加载 MovieLens CTR 教学切片


[data_prepare] 1/1 rows=4200


[train] 0/3 训练 GBDT 并学习叶节点编码


[train] 1/3 编码 GBDT 叶节点


[train] 2/3 训练叶节点 Logistic Regression


...

[inference] 0/1 生成测试集概率


[inference] 1/1


[evaluate] 0/1 计算 AUC 与 LogLoss


[evaluate] 1/1 auc=0.445599 logloss=0.907944


{'profile': 'smoke', 'executed_dataset': 'MovieLens latest-small'}
{'randomly_fabricated_rows': 0, 'auc': 0.4455990920763023, 'logloss': 0.9079436454734647}


## Checks

bounded teaching view 确认树数、叶特征维数、概率范围和 AUC；正式产物来自 runner。

In [8]:
assert train_leaf.shape[1] == gbdt.n_estimators
assert train_leaf_onehot.shape[1] > train_leaf.shape[1]
assert np.all((gbdt_lr_probability >= 0) & (gbdt_lr_probability <= 1))
assert gbdt_lr_auc > .55
print('PASS：GBDT、叶编码、LR 与在线推理链均有效。')

PASS：GBDT、叶编码、LR 与在线推理链均有效。


In [9]:
result_dir = ARTIFACT_ROOT / "results" / "chapter_4"; result_dir.mkdir(parents=True, exist_ok=True)
payload = {"records": [{"algorithm":"GBDT+LR", "task":"CTR", "metric":"AUC ↑", "value":result["auc"],
                       "secondary_metric":"LogLoss ↓", "secondary_value":result.get("logloss"),
                       "status":"executed", "online_inference":"树叶编码 → LR 概率",
                       "source_notebook":"4_5_gbdt_lr"}]}
(result_dir / "gbdt_lr.json").write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("已写出章节指标：gbdt_lr.json")

已写出章节指标：gbdt_lr.json


## Next Steps

与原始特征 LR、FM 对照；加入概率校准曲线；模拟数据漂移并监控叶节点覆盖变化。